# %% [markdown]

 # Grocery Recommendations Prototype – Task 1 & 2

 This notebook covers:
 1. Mining frequent itemsets (Task 1)
 2. Building a baseline collaborative‑filtering recommender (Task 2)

 **Notebook style guide**
 * No large monolithic functions – each step is explicit.
 * Run cells top‑to‑bottom.
 * Adjust paths / parameters as needed.

 ---


# %% [markdown]

 ## 0 Imports & dataset paths


In [1]:
# %%

import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import warnings, datetime as dt, math, itertools
warnings.filterwarnings('ignore')

/var/folders/hp/y1ysddx12bd1c9rpj6fg17s00000gn/T/ipykernel_31851/4183442081.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


# %% [markdown]

 ## 1 Load train / test splits


In [2]:
# %%

train_path = 'Groceries data train.csv'
test_path  = 'Groceries data test.csv'
train_raw  = pd.read_csv(train_path)
test_raw   = pd.read_csv(test_path)

# Debug prints
print("Train columns before strip:", train_raw.columns.tolist())
print("Test columns before strip:", test_raw.columns.tolist())

# Clean column names by stripping whitespace
train_raw.columns = train_raw.columns.str.strip()
test_raw.columns = test_raw.columns.str.strip()

# Debug prints after cleaning
print("\nTrain columns after strip:", train_raw.columns.tolist())
print("Test columns after strip:", test_raw.columns.tolist())

print("\nFirst few rows of train data:")
print(train_raw.head())
print("\nFirst few rows of test data:")
print(test_raw.head())

# Standardize column names
test_raw = test_raw.rename(columns={'user_id': 'User_id'})

# Handle any NA values in User_id
train_raw['User_id'] = train_raw['User_id'].fillna(0)
test_raw['User_id'] = test_raw['User_id'].fillna(0)

train_raw['BasketID'] = train_raw['User_id'].astype(int).astype(str) + '_' + train_raw['Date']
test_raw['BasketID']  = test_raw['User_id'].astype(int).astype(str) + '_' + test_raw['Date']

print('Unique baskets in train:', train_raw['BasketID'].nunique())

Train columns before strip: ['User_id', 'Date', 'itemDescription', 'year', 'month', 'day', 'day_of_week']
Test columns before strip: ['user_id', 'Date', 'itemDescription', 'year', 'month', 'day', 'day_of_week']

Train columns after strip: ['User_id', 'Date', 'itemDescription', 'year', 'month', 'day', 'day_of_week']
Test columns after strip: ['user_id', 'Date', 'itemDescription', 'year', 'month', 'day', 'day_of_week']

First few rows of train data:
   User_id       Date itemDescription    year  month  day  day_of_week
0   2351.0  1/01/2014         cleaner  2014.0    1.0  1.0          2.0
1   2226.0  1/01/2014         sausage  2014.0    1.0  1.0          2.0
2   1922.0  1/01/2014  tropical fruit  2014.0    1.0  1.0          2.0
3   2943.0  1/01/2014      whole milk  2014.0    1.0  1.0          2.0
4   1249.0  1/01/2014    citrus fruit  2014.0    1.0  1.0          2.0

First few rows of test data:
   user_id        Date   itemDescription  year  month  day  day_of_week
0     2889  20/01/20

# %% [markdown]

 ## 4 One‑hot encode baskets for Apriori


In [3]:
# %%

basket_train = (
    train_raw
      .groupby(['BasketID', 'itemDescription'])['itemDescription']
      .count()
      .unstack(fill_value=0)
      .astype(bool)
      .astype(int)
)

print('Basket × item matrix shape:', basket_train.shape)
basket_train.head()

Basket × item matrix shape: (8361, 167)


itemDescription,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
BasketID,,,,,,,,,,,,,,,,,,,,,
1000_24/06/2014,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1001_12/12/2014,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1001_2/07/2014,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1001_20/01/2015,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1002_2/09/2014,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# %% [markdown]

 ## 5 Frequent itemset mining (Apriori)
 * **min_support** set low (e.g. 0.01) – adjust to taste.
 * We keep itemsets up to length 3 to keep things tractable.


In [4]:
# %%

min_support = 0.01
freq_itemsets = apriori(basket_train,
                        min_support=min_support,
                        use_colnames=True,
                        max_len=3,
                        low_memory=True)

freq_itemsets.sort_values('support', ascending=False).head()

,support,itemsets
60,0.130487,(whole milk)
38,0.106686,(other vegetables)
45,0.101902,(rolls/buns)
51,0.094606,(soda)
61,0.078579,(yogurt)


# %% [markdown]

 ### 5.1 Add extra quality metrics
 * **Confidence / lift** via `association_rules`.
 * Custom **importance** score = *support × lift* (feel free to tweak).


In [5]:
# %%

rules = association_rules(freq_itemsets, metric='confidence', min_threshold=0.0)
rules['importance'] = rules['support'] * rules['lift']
rules.sort_values('importance', ascending=False).head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,importance


# %% [markdown]

 ### 5.2 Store a tidy table of pattern → score


In [6]:
# %%

pattern_scores = (
    rules[['antecedents', 'consequents', 'importance']]
      .sort_values('importance', ascending=False)
      .reset_index(drop=True)
)

pattern_scores.head()

,antecedents,consequents,importance


# %% [markdown]

 ## 6 User–item interaction matrix for collaborative filtering
 Binary purchase indicator is enough for implicit feedback.


In [7]:
# %%

user_item = (
    train_raw
      .groupby(['User_id', 'itemDescription'])['itemDescription']
      .count()
      .unstack(fill_value=0)
      .astype(bool)
      .astype(int)
)

print('Users:', user_item.shape[0], '| Items:', user_item.shape[1])

Users: 3493 | Items: 167


# %% [markdown]

 ### 6.1 Compute cosine similarity between users


In [8]:
# %%

user_sim = cosine_similarity(user_item)
user_sim = pd.DataFrame(user_sim,
                        index=user_item.index,
                        columns=user_item.index)

# %% [markdown]

 ## 7 Predict top‑N items for each user (without frequent itemsets)
 *Score = weighted sum of neighbours' purchase indicators.*


In [9]:
# %%

scores_cf = user_sim.dot(user_item)           # (users × items)
scores_cf = scores_cf / user_sim.sum(axis=1).values.reshape(-1,1)
scores_cf.head()

itemDescription,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
User_id,,,,,,,,,,,,,,,,,,,,,
1000.0,0.007913,0.043947,0.003318,0.005574,0.000000,0.000000,0.014528,0.004478,0.051063,0.040892,...,0.005700,0.004160,0.059432,0.095897,0.002565,0.067040,0.021753,0.729488,0.161202,0.010819
1001.0,0.010379,0.048083,0.004324,0.004588,0.000602,0.000322,0.017822,0.004678,0.045942,0.028625,...,0.007603,0.006667,0.050558,0.098863,0.001805,0.060047,0.025006,0.512302,0.166816,0.008057
1002.0,0.007957,0.047398,0.003977,0.004303,0.000000,0.000394,0.017532,0.003914,0.048916,0.034616,...,0.006597,0.006660,0.055352,0.095056,0.002022,0.063317,0.024205,0.574609,0.174813,0.008454
1003.0,0.012331,0.047645,0.005762,0.003521,0.001141,0.001220,0.024429,0.002339,0.052098,0.033797,...,0.009756,0.005793,0.042258,0.093818,0.001990,0.061429,0.028768,0.258312,0.168886,0.007056
1004.0,0.009758,0.047974,0.004561,0.004600,0.000316,0.000507,0.019496,0.003484,0.047972,0.037208,...,0.008367,0.006369,0.049134,0.095641,0.002237,0.060488,0.025546,0.394304,0.171617,0.006576


# %% [markdown]

 ### 7.1 Mask already‑purchased items


In [10]:
# %%

already_owned = user_item.astype(bool)
scores_cf = scores_cf.mask(already_owned, other=np.NINF)

# %% [markdown]

 ### 7.2 Top‑5 recommendations per user


In [11]:
# %%

TopN = 5
top5_cf = (
    scores_cf
      .apply(lambda row: row.nlargest(TopN).index.tolist(), axis=1)
)

top5_cf.head()

User_id
1000.0    [rolls/buns, other vegetables, soda, yogurt, b...
1001.0    [other vegetables, yogurt, bottled water, root...
1002.0    [rolls/buns, soda, yogurt, bottled water, root...
1003.0    [whole milk, other vegetables, soda, yogurt, s...
1004.0    [soda, yogurt, bottled water, bottled beer, wh...
dtype: object

# %% [markdown]

 ## 8 Merge CF with frequent‑pattern recommendations (optional)
 **Heuristic**
 1. For the target user, find patterns whose antecedent ⊆ their basket history.
 2. Add consequent items (not yet purchased) with weight = pattern importance.
 3. Combine CF score + pattern weight (e.g. `score_final = α·CF + (1‑α)·FP`).
 4. Rank and pick top 5.

 *Tune `alpha` on validation data.*


In [12]:
# %%

alpha = 0.7   # 70% CF, 30% pattern‑based – tweak later

# Build a dict for fast user history lookup
user_history = train_raw.groupby('User_id')['itemDescription'].apply(set)

# Example for one user (feel free to loop later)
example_user = user_item.index[0]
history      = user_history[example_user]

# pattern candidates
candidates = []
for _, row in pattern_scores.iterrows():
    if row['antecedents'].issubset(history):
        for itm in row['consequents']:
            if itm not in history:
                candidates.append((itm, row['importance']))
                
fp_df = pd.DataFrame(candidates, columns=['item', 'fp_score']).groupby('item').max()

# CF part for this user
cf_scores_u = scores_cf.loc[example_user].dropna().to_frame('cf_score')

merged = cf_scores_u.join(fp_df, how='outer').fillna(0)
merged['final'] = alpha*merged['cf_score'] + (1-alpha)*merged['fp_score']
merged.sort_values('final', ascending=False).head(TopN)

,cf_score,fp_score,final
itemDescription,,,
rolls/buns,0.216432,0,0.151502
other vegetables,0.213206,0,0.149244
soda,0.189304,0,0.132513
yogurt,0.161202,0,0.112842
bottled water,0.135396,0,0.094778


# %% [markdown]

 ## 9 (Optional) Recency boost
 If user's last purchase date for an itemset is recent, we can up‑weight it.


In [13]:
# %%

# Build a simple recency factor (days since last purchase)
train_raw['Date_dt'] = pd.to_datetime(train_raw['Date'], format='%d/%m/%Y')
last_purchase = train_raw.groupby('User_id')['Date_dt'].max()

def recency_weight(u):
    today = last_purchase.max()
    delta = (today - last_purchase[u]).days + 1
    return 1 / math.log1p(delta)

rec_w = recency_weight(example_user)
print('Recency weight for user', example_user, ':', rec_w)

Recency weight for user 1000.0 : 0.18668606248878736


# %% [markdown]

 ## 10 Mini CLI for manual testing


In [14]:
# %%

while True:
    try:
        uid = int(input('Enter user_id (or 0 to quit): '))
    except ValueError:
        print('Not a number.')
        continue
    if uid == 0:
        break
    mode = input('Type "with" for CF+patterns, anything else for CF only: ')
    
    if uid not in user_item.index:
        print('Cold‑start user – recommend top popular items')
        popular = train_raw['itemDescription'].value_counts().head(TopN).index.tolist()
        print(popular)
        continue
    
    cf_row   = scores_cf.loc[uid].dropna()
    cf_row   = cf_row.mask(user_item.loc[uid].astype(bool), other=np.NINF)
    
    if mode.strip().lower() != 'with':
        print(cf_row.nlargest(TopN).index.tolist())
        continue
    
    hist     = user_history[uid]
    cand     = []
    for _, r in pattern_scores.iterrows():
        if r['antecedents'].issubset(hist):
            for it in r['consequents']:
                if it not in hist:
                    cand.append((it, r['importance']))
    fp_part = pd.DataFrame(cand, columns=['item', 'fp_score']).groupby('item').max()
    merged  = cf_row.to_frame('cf_score').join(fp_part, how='outer').fillna(0)
    merged['final'] = alpha*merged['cf_score'] + (1-alpha)*merged['fp_score']
    print(merged.sort_values('final', ascending=False).head(TopN).index.tolist())

['other vegetables', 'yogurt', 'bottled water', 'root vegetables', 'shopping bags']
['rolls/buns', 'soda', 'yogurt', 'bottled water', 'root vegetables']
['soda', 'yogurt', 'bottled water', 'bottled beer', 'whipped/sour cream']
